[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/02-python-for-data/notebooks/03-groupby-merge-excel.ipynb)

# 03 · groupby, merge and Excel reports

**What you will learn**
- answer "per group" questions with `groupby` and named aggregation
- lay a summary out as a grid with `pivot_table`
- join tables with `merge` (inner vs left) and check the result with `indicator=True`
- write a multi-sheet Excel report with `ExcelWriter`, then style it with **openpyxl**

### Finding the practice data

The practice files live in the repo folders `data/` (CSV) and `excel/` (Excel).
The helper below looks for them one folder up from this notebook (that is where they are when you
run the notebook from the repo). If they are not there (for example in Google Colab), it downloads
them from GitHub into the current folder. You do not need to change anything.

In [1]:
from pathlib import Path
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/vvvvvivekkk/AI-Engineer/main/02-python-for-data/"

def data_path(rel_path):
    """Return a local path to a practice file such as 'data/sales.csv'."""
    for candidate in (Path("..") / rel_path, Path(rel_path)):
        if candidate.exists():
            return candidate
    # Not found locally -> download it from GitHub (works in Colab).
    target = Path(rel_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW_BASE + rel_path, target)
    return target

print(data_path("data/sales.csv"))

../data/sales.csv


## 1. Load the three related tables

- `sales`: one row per order, with `customer_id` and `product_id`
- `products`: product name, category, price
- `customers`: name, city, segment

In [2]:
import pandas as pd

sales = pd.read_csv(data_path("data/sales.csv"), parse_dates=["order_date"])
products = pd.read_csv(data_path("data/products.csv"))
customers = pd.read_csv(data_path("data/customers.csv"), parse_dates=["signup_date"])

sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount_pct"] / 100)
sales["month"] = sales["order_date"].dt.to_period("M")
print(sales.shape, products.shape, customers.shape)

(600, 10) (12, 4) (60, 5)


## 2. A first groupby

"Total revenue **per** channel." Split by channel, sum the revenue in each group, combine into a small Series.

In [3]:
sales.groupby("channel")["revenue"].sum().round(0)

channel
App        335718.0
Store      183886.0
Website    508365.0
Name: revenue, dtype: float64

## 3. Several numbers at once (named aggregation)

The pattern is `new_column=(existing_column, function)`. Common functions: `"sum"`, `"mean"`, `"count"`, `"min"`, `"max"`, `"median"`, `"nunique"` (number of distinct values).

In [4]:
by_channel = sales.groupby("channel").agg(
    orders=("order_id", "count"),
    customers=("customer_id", "nunique"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
).round(0).sort_values("revenue", ascending=False)
by_channel

,orders,customers,revenue,avg_order
channel,,,,
Website,291,51,508365.0,1747.0
App,209,49,335718.0,1606.0
Store,100,38,183886.0,1839.0


## 4. Grouping by two columns

Pass a list of columns. The result has a two-level index (month, then channel). `.unstack()` turns the inner level into columns, which is easier to read.

In [5]:
monthly_channel = sales.groupby(["month", "channel"])["revenue"].sum().round(0)
print(monthly_channel.head(6))
monthly_channel.unstack()

month    channel
2026-01  App        54578.0
         Store      17386.0
         Website    80632.0
2026-02  App        38660.0
         Store      32586.0
         Website    46430.0
Name: revenue, dtype: float64


channel,App,Store,Website
month,,,
2026-01,54578.0,17386.0,80632.0
2026-02,38660.0,32586.0,46430.0
2026-03,79022.0,29508.0,75611.0
2026-04,26841.0,51802.0,88359.0
2026-05,67286.0,24525.0,106015.0
2026-06,69333.0,28079.0,111317.0


## 5. Pivot tables

`pivot_table` does the same in one call, like Excel's PivotTable. `margins=True` adds an "All" total row and column.

In [6]:
pivot = pd.pivot_table(sales, index="month", columns="channel", values="revenue",
                       aggfunc="sum", margins=True, margins_name="Total").round(0)
pivot

channel,App,Store,Website,Total
month,,,,
2026-01,54578.0,17386.0,80632.0,152596.0
2026-02,38660.0,32586.0,46430.0,117676.0
2026-03,79022.0,29508.0,75611.0,184141.0
2026-04,26841.0,51802.0,88359.0,167002.0
2026-05,67286.0,24525.0,106015.0,197826.0
2026-06,69333.0,28079.0,111317.0,208729.0
Total,335718.0,183886.0,508365.0,1027970.0


## 6. Merge sales with products

`merge(..., on="product_id", how="left")` keeps every sale and adds the product's name and category.
We drop the duplicate price column from products first (sales already has `unit_price`).

In [7]:
print("rows before:", len(sales))
full = sales.merge(products.drop(columns="unit_price"), on="product_id", how="left")
print("rows after: ", len(full))
full[["order_id", "product_id", "product_name", "category", "revenue"]].head()

rows before: 600
rows after:  600


,order_id,product_id,product_name,category,revenue
0,O10001,P004,A5 Notebook (5 pack),Stationery,747.00
1,O10002,P002,Bluetooth Headphones,Electronics,4998.00
2,O10003,P005,Gel Pen Set,Stationery,894.00
3,O10004,P004,A5 Notebook (5 pack),Stationery,423.30
4,O10005,P005,Gel Pen Set,Stationery,707.75


## 7. Merge with customers and look for problems

`indicator=True` adds a `_merge` column telling us whether each row matched (`both`) or only existed in the left table (`left_only`).
Three orders come from customer `C099`, who is missing from the customer table. A left join keeps them (with blank city); an inner join would silently drop them.

In [8]:
check = full.merge(customers, on="customer_id", how="left", indicator=True)
print(check["_merge"].value_counts())
print()
print(check.loc[check["_merge"] == "left_only", ["order_id", "customer_id", "customer_name", "city"]])

_merge
both          597
left_only       3
right_only      0
Name: count, dtype: int64

    order_id customer_id customer_name city
57    O10058        C099           NaN  NaN
301   O10302        C099           NaN  NaN
488   O10489        C099           NaN  NaN


Compare the row counts of a left join and an inner join. The difference is exactly the unmatched orders.

In [9]:
left = full.merge(customers, on="customer_id", how="left")
inner = full.merge(customers, on="customer_id", how="inner")
print("left join rows: ", len(left))
print("inner join rows:", len(inner))
full = left  # keep the left join for the rest of the notebook
full["city"] = full["city"].fillna("Unknown")

left join rows:  600
inner join rows: 597


## 8. Questions that need the merged table

Now "revenue per category" and "revenue per city" are one groupby each.

In [10]:
by_category = full.groupby("category").agg(units=("quantity", "sum"), revenue=("revenue", "sum")).round(0)
by_city = full.groupby("city").agg(orders=("order_id", "count"), revenue=("revenue", "sum")).round(0)
print(by_category.sort_values("revenue", ascending=False), "\n")
print(by_city.sort_values("revenue", ascending=False))

             units   revenue
category                    
Electronics    296  368671.0
Clothing       319  312842.0
Home           234  183210.0
Stationery     739  163245.0 

           orders   revenue
city                       
Mumbai        155  282426.0
Bengaluru     155  275357.0
Hyderabad     134  221631.0
Chennai        86  132562.0
Pune           45   62769.0
Delhi          22   49517.0
Unknown         3    3707.0


## 9. Top customers

Group by customer name, sum, sort, keep the top 5. `reset_index()` turns the group labels back into a normal column.

In [11]:
top_customers = (full.groupby(["customer_id", "customer_name"])["revenue"].sum()
                 .sort_values(ascending=False).head(5).round(0).reset_index())
top_customers

,customer_id,customer_name,revenue
0,C057,Vikram Bose,92420.0
1,C003,Ananya Menon,67960.0
2,C036,Rahul Iyer,59218.0
3,C048,Sneha Sharma,51060.0
4,C008,Diya Rao,47702.0


## 10. Write a multi-sheet Excel report

`pd.ExcelWriter` keeps one workbook open while we add several sheets. The `with` block saves and closes the file at the end.
`index=False` skips the row numbers when they mean nothing.

In [12]:
from pathlib import Path
out = Path("outputs")
out.mkdir(exist_ok=True)
report_path = out / "sales_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    by_channel.to_excel(writer, sheet_name="By channel")
    by_category.to_excel(writer, sheet_name="By category")
    by_city.to_excel(writer, sheet_name="By city")
    pivot.astype(float).to_excel(writer, sheet_name="Month x channel")
    top_customers.to_excel(writer, sheet_name="Top customers", index=False)
    full.drop(columns=["month"]).to_excel(writer, sheet_name="Raw data", index=False)

print("written:", report_path)

written: outputs/sales_report.xlsx


## 11. Style it with openpyxl

pandas writes values only. **openpyxl** can open the file and change its look: bold headers, column widths, number formats, and even live formulas.
Here we style the "By category" sheet and add a formula column that computes each category's share of revenue. Excel will calculate the formula when the file is opened.

In [13]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

wb = load_workbook(report_path)
ws = wb["By category"]

header_fill = PatternFill("solid", fgColor="1B263B")
for cell in ws[1]:                       # row 1 = header row
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = header_fill

ws["D1"] = "share_of_revenue"
ws["D1"].font = Font(bold=True, color="FFFFFF")
ws["D1"].fill = header_fill
last = ws.max_row
for r in range(2, last + 1):
    ws[f"D{r}"] = f"=C{r}/SUM($C$2:$C${last})"   # a live Excel formula
    ws[f"D{r}"].number_format = "0.0%"
    ws[f"C{r}"].number_format = "#,##0"

for col, width in zip("ABCD", [16, 10, 14, 18]):
    ws.column_dimensions[col].width = width

wb.save(report_path)
print("sheets:", wb.sheetnames)

sheets: ['By channel', 'By category', 'By city', 'Month x channel', 'Top customers', 'Raw data']


## 12. Read it back

When pandas reads a cell that holds a formula written by openpyxl, it gets nothing useful (the file has never been calculated by Excel).
That is why we compute numbers in pandas and only use formulas for things a human might want to change in Excel.

In [14]:
back = pd.read_excel(report_path, sheet_name=None)
for name, table in back.items():
    print(f"{name:16s} {table.shape}")
back["By category"]

By channel       (3, 5)
By category      (4, 4)
By city          (7, 3)
Month x channel  (7, 5)
Top customers    (5, 3)
Raw data         (600, 15)


,category,units,revenue,share_of_revenue
0,Clothing,319,312842,NaN
1,Electronics,296,368671,NaN
2,Home,234,183210,NaN
3,Stationery,739,163245,NaN


## Recap

- `groupby(...).agg(name=(col, func))` answers "per group" questions.
- `pivot_table` makes an Excel-style grid.
- `merge(on=..., how="left")` adds columns from another table; always count rows and use `indicator=True` to spot unmatched keys.
- `ExcelWriter` writes several sheets; `openpyxl` styles them and adds formulas.

Next: **04 · mini EDA**.